# Pipeline Settings

Edit the values in the next cell, then **run all cells**.

## LLM provider

| `LLM_PROVIDER` | When to use | Setup |
|----------------|-------------|-------|
| **`"openrouter"`** | Cloud | `export OPENROUTER_API_KEY=sk-or-...` |
| **`"ollama"`** | Local (recommended for MIMIC) | `ollama pull qwen2.5:7b` + `ollama serve` |

## MIMIC + OpenRouter

Credentialed MIMIC data requires **zero data retention** on third-party APIs. When using OpenRouter:

- Keep **`OPENROUTER_ZDR: true`** (default) — sends `provider: { zdr: true }` on every request
- Also disable prompt logging in your [OpenRouter privacy settings](https://openrouter.ai/settings/privacy)
- Local **Ollama** remains the safest option per [PhysioNet guidance](https://physionet.org/news/post/llm-responsible-use/)

**After changing settings:** restart the kernel in stage notebooks 2–3.

In [ ]:
import json
from pathlib import Path

SETTINGS = {
    "TEST_MODE": False,
    "PHYSIONET_ROOT": "/Users/narenkhatwani/Desktop/physionet.org/files",

    "LLM_PROVIDER": "openrouter",  # "openrouter" | "ollama"

    "OPENROUTER_API_KEY": None,
    # MIMIC credentialed data: keep True when using OpenRouter
    "OPENROUTER_ZDR": True,

    "OLLAMA_BASE_URL": "http://localhost:11434",
    "OLLAMA_MODEL": "qwen2.5:7b",

    "RANDOM_SEED": 42,
    "LLM_REQUEST_DELAY_SECONDS": 3.0,
}

provider = str(SETTINGS.get("LLM_PROVIDER", "openrouter")).lower().strip()
if provider not in ("openrouter", "ollama"):
    raise ValueError('LLM_PROVIDER must be "openrouter" or "ollama"')

out = Path("settings.json")
out.write_text(json.dumps(SETTINGS, indent=2), encoding="utf-8")
print(f"Saved → {out.resolve()}")
print(f"LLM_PROVIDER = {provider!r}")
if provider == "openrouter":
    zdr = SETTINGS.get("OPENROUTER_ZDR", True)
    print(f"  → OpenRouter / qwen/qwen-2.5-7b-instruct | ZDR={'on' if zdr else 'OFF'}")
else:
    print(f"  → Ollama / {SETTINGS['OLLAMA_MODEL']} at {SETTINGS['OLLAMA_BASE_URL']}")

## Verify LLM connection

In [ ]:
import sys
from pathlib import Path

NB = Path.cwd()
sys.path.insert(0, str(NB if (NB / "pipeline.py").exists() else NB))

import importlib
import pipeline
importlib.reload(pipeline)

pipeline.print_pipeline_banner()
cfg = pipeline.get_llm_config()
if pipeline.LLM_PROVIDER == "openrouter":
    print(f"Request ZDR  : {cfg.openrouter_zdr} (provider.zdr in API payload)")
ok, msg = pipeline.check_llm(cfg)
if ok:
    print(f"LLM ready — {cfg.method_prefix()}: {msg}")
else:
    print(f"LLM not ready — {msg}")
    if pipeline.LLM_PROVIDER == "openrouter":
        print("  Fix: export OPENROUTER_API_KEY=sk-or-...")
    else:
        print("  Fix: ollama serve && ollama pull qwen2.5:7b")